# NB09 — Blog Visualizations

## Overview

This notebook creates all Plotly charts for the blog post on documentation gaps and revenue opportunities. Each chart is saved as a standalone HTML div for embedding in the blog.

### Charts to Create:

1. **Chart 1**: CMI Distribution by Ownership Type (Box plot)
2. **Chart 2**: Severity Tier Distribution – National (Stacked bar)
3. **Chart 3**: Documentation Gap Score Distribution (Histogram)
4. **Chart 4**: Revenue Opportunity by Hospital Size (Bar chart)
5. **Chart 5**: CMI vs Documentation Gap Score (Scatter plot)
6. **Chart 6**: Revenue Opportunity by State (Bar chart – Top 15)
7. **Chart 7**: Severity Tier Payment Uplift (Grouped bar chart)

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

# Define project root and paths
PROJECT_ROOT = Path.cwd().parents[1]  # Navigate up to project root

# Load the full hospital dataset from NB06 (has CMI, severity, beds, etc.)
hospital_data = pd.read_csv(
    PROJECT_ROOT / 'data' / 'outputs' / 'nb06_peer_benchmarks' / 'hospital_with_benchmarks.csv',
    dtype={'ccn': str}
)
hospital_data['ccn'] = hospital_data['ccn'].str.zfill(6)

# Load revenue impact from NB08 (has gap scores, revenue estimates)
revenue_only = pd.read_csv(
    PROJECT_ROOT / 'data' / 'outputs' / 'nb08_revenue_impact' / 'hospital_revenue_impact.csv',
    dtype={'ccn': str}
)
revenue_only['ccn'] = revenue_only['ccn'].str.zfill(6)

# Merge: take full hospital data + revenue columns from NB08
revenue_cols_to_add = [c for c in revenue_only.columns if c not in hospital_data.columns]
revenue_cols_to_add.append('ccn')  # need join key
revenue_data = hospital_data.merge(revenue_only[revenue_cols_to_add], on='ccn', how='left')

# Load severity tier payment data
severity_data = pd.read_csv(
    PROJECT_ROOT / 'data' / 'outputs' / 'nb02_drg_cmi' / 'severity_tier_payments.csv'
)

print(f"Loaded hospital+revenue data: {revenue_data.shape}")
print(f"Loaded severity data: {severity_data.shape}")
print(f"\nKey columns available: cmi={('cmi' in revenue_data.columns)}, "
      f"ownership_category={('ownership_category' in revenue_data.columns)}, "
      f"doc_gap_score={('doc_gap_score' in revenue_data.columns)}, "
      f"estimated_annual_revenue_opportunity={('estimated_annual_revenue_opportunity' in revenue_data.columns)}")

# Define color palette
COLORS = {
    'primary': '#2E86AB',    # blue
    'secondary': '#A23B72',  # magenta
    'accent': '#F18F01',     # orange
    'success': '#2CA58D',    # green
    'danger': '#E15554',     # red
    'light': '#F5F5F5',
    'dark': '#1B1B2F',
}

# Create output directory
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs' / 'blog_charts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nOutput directory: {OUTPUT_DIR}")

def save_chart(fig, filename):
    """Save chart as HTML div for blog embedding."""
    html = fig.to_html(full_html=False, include_plotlyjs='cdn')
    path = OUTPUT_DIR / f'{filename}.html'
    with open(path, 'w') as f:
        f.write(html)
    print(f'Saved: {path.name}')

Loaded hospital+revenue data: (3280, 75)
Loaded severity data: (313, 9)

Key columns available: cmi=True, ownership_category=True, doc_gap_score=True, estimated_annual_revenue_opportunity=True

Output directory: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/blog_charts


## Chart 1: CMI Distribution by Ownership Type

In [2]:
# Prepare data: drop rows with missing CMI or ownership_category
chart1_data = revenue_data[['cmi', 'ownership_category']].dropna()

# Create box plot
fig1 = px.box(
    chart1_data,
    x='ownership_category',
    y='cmi',
    color='ownership_category',
    title='Case Mix Index Distribution by Hospital Ownership Type',
    labels={'cmi': 'CMI', 'ownership_category': 'Ownership Type'},
    template='plotly_white',
    height=500,
    width=900,
    color_discrete_sequence=[
        COLORS['primary'],
        COLORS['secondary'],
        COLORS['accent'],
        COLORS['success']
    ]
)

fig1.update_layout(
    showlegend=False,
    font=dict(size=12),
    xaxis_title='Ownership Type',
    yaxis_title='Case Mix Index (CMI)',
    title_font_size=16
)

fig1.show()
save_chart(fig1, 'chart1_cmi_by_ownership')

Saved: chart1_cmi_by_ownership.html


## Chart 2: Severity Tier Distribution (National)

In [3]:
# Create national severity tier distribution
# The hospital-level data has pct_without_CC, pct_with_CC, pct_with_MCC (already in %)
if 'pct_without_CC' in revenue_data.columns:
    without_cc = revenue_data['pct_without_CC'].dropna().mean()
    with_cc = revenue_data['pct_with_CC'].dropna().mean()
    with_mcc = revenue_data['pct_with_MCC'].dropna().mean()
elif 'without_cc' in revenue_data.columns:
    without_cc = revenue_data['without_cc'].mean() * 100
    with_cc = revenue_data['with_cc'].mean() * 100
    with_mcc = revenue_data['with_mcc'].mean() * 100
else:
    total_without = severity_data['without_CC'].sum()
    total_cc = severity_data['with_CC'].sum()
    total_mcc = severity_data['with_MCC'].sum()
    grand_total = total_without + total_cc + total_mcc
    without_cc = total_without / grand_total * 100
    with_cc = total_cc / grand_total * 100
    with_mcc = total_mcc / grand_total * 100

print(f"National severity distribution:")
print(f"  Without CC/MCC: {without_cc:.1f}%")
print(f"  With CC: {with_cc:.1f}%")
print(f"  With MCC: {with_mcc:.1f}%")

# Use a vertical bar chart so each tier is clearly visible
tiers = ['Without CC/MCC', 'With CC', 'With MCC']
values = [without_cc, with_cc, with_mcc]
colors = [COLORS['danger'], COLORS['accent'], COLORS['primary']]

fig2 = go.Figure(data=[
    go.Bar(
        x=tiers,
        y=values,
        marker_color=colors,
        text=[f'{v:.1f}%' for v in values],
        textposition='outside',
        textfont=dict(size=14, color='#333')
    )
])

fig2.update_layout(
    title='National Medicare Severity Tier Distribution',
    yaxis_title='Percentage of Cases (%)',
    xaxis_title='Severity Tier',
    template='plotly_white',
    height=500,
    width=900,
    showlegend=False,
    font=dict(size=12),
    title_font_size=16,
    yaxis_range=[0, max(values) * 1.15]
)

fig2.show()
save_chart(fig2, 'chart2_severity_national')

National severity distribution:
  Without CC/MCC: 4.5%
  With CC: 17.5%
  With MCC: 63.9%


Saved: chart2_severity_national.html


## Chart 3: Documentation Gap Score Distribution

In [4]:
# Prepare data: drop rows with missing gap score or gap tier
chart3_data = revenue_data[['doc_gap_score', 'gap_tier']].dropna()

# Define color map for gap tiers
gap_tier_colors = {
    'High': COLORS['danger'],
    'Medium': COLORS['accent'],
    'Low': COLORS['success']
}

# Create histogram
fig3 = px.histogram(
    chart3_data,
    x='doc_gap_score',
    color='gap_tier',
    nbins=30,
    title='Distribution of Documentation Gap Scores Across U.S. Hospitals',
    labels={'doc_gap_score': 'Documentation Gap Score', 'gap_tier': 'Gap Tier'},
    template='plotly_white',
    height=500,
    width=900,
    color_discrete_map=gap_tier_colors,
    barmode='overlay'
)

fig3.update_traces(opacity=0.75)
fig3.update_layout(
    font=dict(size=12),
    xaxis_title='Documentation Gap Score',
    yaxis_title='Number of Hospitals',
    title_font_size=16,
    hovermode='x unified'
)

fig3.show()
save_chart(fig3, 'chart3_gap_score_distribution')

Saved: chart3_gap_score_distribution.html


## Chart 4: Revenue Opportunity by Hospital Size

In [5]:
# Prepare data: drop rows with missing values
chart4_data = revenue_data[['bed_size_tier', 'estimated_annual_revenue_opportunity']].dropna()

# Check actual bed size tier values
print(f"Bed size tiers in data: {sorted(chart4_data['bed_size_tier'].unique())}")

# Define logical order for bed size tiers (use whatever values actually exist)
all_possible_orders = ['1-24', '25-99', '100-199', '200-299', '200-399', '300-399', '400-499', '400-599', '500-599', '600+']
actual_tiers = chart4_data['bed_size_tier'].unique()
bed_size_order = [t for t in all_possible_orders if t in actual_tiers]

# Calculate mean revenue opportunity by bed size
revenue_by_size = chart4_data.groupby('bed_size_tier')['estimated_annual_revenue_opportunity'].mean().reset_index()

# Reorder by logical bed size progression
revenue_by_size['bed_size_tier'] = pd.Categorical(
    revenue_by_size['bed_size_tier'], 
    categories=bed_size_order, 
    ordered=True
)
revenue_by_size = revenue_by_size.sort_values('bed_size_tier').dropna(subset=['bed_size_tier'])

# Create bar chart
fig4 = px.bar(
    revenue_by_size,
    x='bed_size_tier',
    y='estimated_annual_revenue_opportunity',
    title='Average Annual Revenue Opportunity by Hospital Size',
    labels={'bed_size_tier': 'Hospital Bed Size', 'estimated_annual_revenue_opportunity': 'Average Annual Revenue Opportunity'},
    template='plotly_white',
    height=500,
    width=900,
    color_discrete_sequence=[COLORS['primary']]
)

# Format y-axis as currency
fig4.update_yaxes(tickformat='$,.0f')

# hovertemplate goes on traces, not layout
fig4.update_traces(hovertemplate='<b>%{x}</b><br>Revenue Opportunity: $%{y:,.0f}<extra></extra>')

fig4.update_layout(
    showlegend=False,
    font=dict(size=12),
    xaxis_title='Hospital Bed Size',
    yaxis_title='Average Annual Revenue Opportunity',
    title_font_size=16
)

fig4.show()
save_chart(fig4, 'chart4_revenue_by_size')

Bed size tiers in data: ['1-24', '100-199', '200-399', '25-99', '400-599', '600+']


Saved: chart4_revenue_by_size.html


## Chart 5: CMI vs Documentation Gap Score

In [6]:
# Prepare data: drop rows with missing values
chart5_data = revenue_data[['cmi', 'doc_gap_score', 'ownership_category', 'beds', 'hospital_name', 'state']].dropna(subset=['cmi', 'doc_gap_score', 'ownership_category'])

# Cap bed size for visualization (capped at 1000)
chart5_data['beds_capped'] = chart5_data['beds'].fillna(100).clip(upper=1000)

# Create scatter plot
fig5 = px.scatter(
    chart5_data,
    x='cmi',
    y='doc_gap_score',
    color='ownership_category',
    size='beds_capped',
    hover_data=['hospital_name', 'state', 'beds'],
    title='CMI vs Documentation Gap Score',
    labels={'cmi': 'Case Mix Index (CMI)', 'doc_gap_score': 'Documentation Gap Score', 'ownership_category': 'Ownership Type'},
    template='plotly_white',
    height=600,
    width=900,
    size_max=30,
    color_discrete_sequence=[COLORS['primary'], COLORS['secondary'], COLORS['accent'], COLORS['success']]
)

fig5.update_layout(
    font=dict(size=12),
    xaxis_title='Case Mix Index (CMI)',
    yaxis_title='Documentation Gap Score',
    title_font_size=16,
    hovermode='closest'
)

fig5.show()
save_chart(fig5, 'chart5_cmi_vs_gap')

Saved: chart5_cmi_vs_gap.html


## Chart 6: Revenue Opportunity by State (Top 15)

In [7]:
# Prepare data: drop rows with missing values
chart6_data = revenue_data[['state', 'estimated_annual_revenue_opportunity']].dropna()

# Calculate total revenue opportunity by state and get top 15
revenue_by_state = chart6_data.groupby('state')['estimated_annual_revenue_opportunity'].sum().reset_index()
revenue_by_state = revenue_by_state.sort_values('estimated_annual_revenue_opportunity', ascending=False).head(15)

# Create bar chart
fig6 = px.bar(
    revenue_by_state,
    x='estimated_annual_revenue_opportunity',
    y='state',
    orientation='h',
    title='Total Documentation Gap Revenue Opportunity by State (Top 15)',
    labels={'estimated_annual_revenue_opportunity': 'Total Annual Revenue Opportunity', 'state': 'State'},
    template='plotly_white',
    height=600,
    width=900,
    color_discrete_sequence=[COLORS['primary']]
)

# Format x-axis as currency
fig6.update_xaxes(tickformat='$,.0f')

# hovertemplate goes on traces, not layout
fig6.update_traces(hovertemplate='<b>%{y}</b><br>Revenue Opportunity: $%{x:,.0f}<extra></extra>')

fig6.update_layout(
    showlegend=False,
    font=dict(size=12),
    xaxis_title='Total Annual Revenue Opportunity',
    yaxis_title='State',
    title_font_size=16
)

fig6.show()
save_chart(fig6, 'chart6_revenue_by_state')

Saved: chart6_revenue_by_state.html


## Chart 7: Severity Tier Payment Uplift

In [8]:
# Prepare data: drop rows with missing values
chart7_data = severity_data[['drg_family', 'cc_uplift', 'mcc_uplift']].dropna()

# Get top 10 DRG families by mcc_uplift
chart7_data = chart7_data.nlargest(10, 'mcc_uplift').copy()

# Shorten DRG family names for readability
def shorten_drg(name):
    replacements = {
        'INTRACRANIAL VASCULAR PROCEDURES WITH PRINCIPAL DIAGNOSIS HEMORRHAGE': 'Intracranial Vascular (Hemorrhage)',
        'OTHER O.R. PROCEDURES FOR MULTIPLE SIGNIFICANT TRAUMA': 'Other OR Proc (Multiple Trauma)',
        'LYMPHOMA AND LEUKEMIA WITH MAJOR O.R. PROCEDURES': 'Lymphoma/Leukemia (Major OR)',
        'SKIN GRAFT FOR SKIN ULCER OR CELLULITIS': 'Skin Graft (Ulcer/Cellulitis)',
        'ACUTE LEUKEMIA': 'Acute Leukemia',
        'SKIN GRAFT, MALIGNANCY, INFECTION OR EXTENSIVE FUSIONS': 'Skin Graft (Malignancy/Infection)',
        'SKIN GRAFT EXCEPT FOR SKIN ULCER OR CELLULITIS': 'Skin Graft (Other)',
        'CARDIOTHORACIC PROCEDURES WITH CARDIAC CATHETERIZATION': 'Cardiothoracic w/ Cath',
        'MUSCULOSKELETAL AND CONNECTIVE TISSUE DISORDERS': 'Musculoskeletal Disorders',
        'CIRCULATORY SYSTEM DISORDERS': 'Circulatory Disorders',
        'AMPUTATION FOR CIRCULATORY SYSTEM DISORDERS EXCEPT UPPER LIMB AND TOE': 'Amputation (Circulatory)',
        'AND OTHER MAJOR CARDIOTHORACIC PROCEDURES': 'Major Cardiothoracic Proc',
        'CERVICAL WITH SPINAL CURVATURE': 'Cervical Spinal Curvature',
    }
    name_upper = name.upper().strip()
    for long, short in replacements.items():
        if long in name_upper:
            return short
    # Fallback: title case and truncate
    short = name.title()
    if len(short) > 35:
        short = short[:32] + '...'
    return short

chart7_data['drg_short'] = chart7_data['drg_family'].apply(shorten_drg)

# Sort ascending so largest is at top of horizontal bar chart
chart7_data = chart7_data.sort_values('mcc_uplift', ascending=True)

# Create horizontal grouped bar chart
fig7 = go.Figure(data=[
    go.Bar(name='CC Uplift', y=chart7_data['drg_short'], x=chart7_data['cc_uplift'],
           marker_color=COLORS['accent'], orientation='h'),
    go.Bar(name='MCC Uplift', y=chart7_data['drg_short'], x=chart7_data['mcc_uplift'],
           marker_color=COLORS['primary'], orientation='h')
])

fig7.update_layout(
    barmode='group',
    title='Payment Uplift per Severity Tier Shift — Top 10 DRG Families',
    xaxis_title='Payment Uplift ($)',
    template='plotly_white',
    height=600,
    width=900,
    font=dict(size=11),
    title_font_size=16,
    hovermode='y unified',
    margin=dict(l=220),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig7.update_xaxes(tickformat='$,.0f')

fig7.show()
save_chart(fig7, 'chart7_severity_uplift')

Saved: chart7_severity_uplift.html


In [9]:
# Summary: List all saved chart files
saved_files = sorted(OUTPUT_DIR.glob('*.html'))

print("\n" + "="*60)
print("BLOG CHART CREATION COMPLETE")
print("="*60)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"Total charts saved: {len(saved_files)}")
print("\nSaved files:")
for i, file in enumerate(saved_files, 1):
    size_kb = file.stat().st_size / 1024
    print(f"  {i}. {file.name} ({size_kb:.1f} KB)")
print("\n" + "="*60)


BLOG CHART CREATION COMPLETE

Output directory: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/blog_charts
Total charts saved: 11

Saved files:
  1. chart10_model_comparison.html (9.2 KB)
  2. chart11_roc_curve.html (19.5 KB)
  3. chart1_cmi_by_ownership.html (87.4 KB)
  4. chart2_severity_national.html (7.9 KB)
  5. chart3_gap_score_distribution.html (44.4 KB)
  6. chart4_revenue_by_size.html (8.2 KB)
  7. chart5_cmi_vs_gap.html (251.7 KB)
  8. chart6_revenue_by_state.html (8.3 KB)
  9. chart7_severity_uplift.html (8.8 KB)
  10. chart8_ols_coefficients.html (10.3 KB)
  11. chart9_ols_diagnostics.html (275.2 KB)

